# Entity Linking and Information Retrieval Debugging with DeepFix

This tutorial demonstrates how to use DeepFix to diagnose and debug **Information Retrieval (IR)** and **Entity Linking** models. In these tasks:
- **Queries** (e.g., search queries or text mentions) are matched against a large corpus of **Documents** or **Entities**.
- The goal is to retrieve the most relevant documents/entities for each query.
- Evaluating these systems involves analyzing query-document pairs, relevance labels, and retrieval ranks/scores.

DeepFix provides native support for IR and Entity Linking datasets via the `InformationRetrievalDataset` class, allowing you to ingest your data, wrap your model with `LlamaindexModel`, and get automated, detailed diagnostic reports (covering data drift, annotation quality, and retrieval performance).

In [ ]:
import os
import pandas as pd
import numpy as np
import pyterrier as pt

from deepfix_sdk import DeepFixClient
from deepfix_sdk.ir import InformationRetrievalDataset, LlamaindexModel

In [ ]:
# Set up your DeepFix API key.
# Get your API key by signing up at https://deepfix.delcaux.com
os.environ["DEEPFIX_API_KEY"] = ""

In [ ]:
# Initialize the DeepFix client
client = DeepFixClient(timeout=300)

## Loading and Preparing the Dataset

We will load a subset of the **BEIR scidocs** dataset using `pyterrier`.
To evaluate the retrieval model, we will:
1. Load queries (topics), ground truth relevance labels (qrels), and documents (corpus).
2. Prepare an `InformationRetrievalDataset` and split it into train and test splits.
3. Index corpus documents and generate real retrievals and embeddings using `LlamaindexModel`.

In [ ]:
def load_ir_data(subset_queries: int = 10, lancedb_index_dir: str = "./.lancedb"):
    """Load BEIR scidocs data using PyTerrier and prepare IR datasets with real LlamaindexModel retrievals."""
    name = "irds:beir/scidocs"
    dataset = pt.get_dataset(name)

    # 1. Get all topics and qrels, subset for fast execution
    all_topics = dataset.get_topics(variant="text")
    all_qrels = dataset.get_qrels()

    qid_subset = all_topics["qid"].unique()[:subset_queries]
    topics_df = all_topics[all_topics["qid"].isin(qid_subset)].copy()
    qrels_df = all_qrels[all_qrels["qid"].isin(qid_subset)].copy()

    needed_docnos = set(qrels_df["docno"].astype(str).unique())
    parent_corpus_iter = dataset.get_corpus_iter

    def subset_corpus_iter():
        count = 0
        for doc in parent_corpus_iter():
            if str(doc["docno"]) in needed_docnos:
                yield doc
                count += 1
                if count >= len(needed_docnos):
                    break

    # 2. Build a single dataset, then split using stratified sampling on labels
    ir_ds = InformationRetrievalDataset(
        dataset_name=name,
        topics=topics_df,
        enable_embedding_pca=True,
        embedding_pca_components=200,
        qrels=qrels_df,
        corpus_iter=subset_corpus_iter,
    )

    train_ir_ds, test_ir_ds = ir_ds.split(train_size=0.7, random_state=42)

    # 3. Generate real retrievals and embeddings using LlamaindexModel
    model = LlamaindexModel(
        dataset=ir_ds,
        load_if_exists=True,
        lancedb_index_dir=lancedb_index_dir,
        top_k=5,
        retrieval_mode="dense",
    )
    model.fit()
    train_ir_ds.set_predictions(model.retrieve_dataframe(train_ir_ds))
    test_ir_ds.set_predictions(model.retrieve_dataframe(test_ir_ds))

    # 4. Set embeddings for diagnostic suites using LlamaindexModel
    train_ir_ds.set_embeddings(model.get_embedding)
    test_ir_ds.set_embeddings(model.get_embedding)

    return train_ir_ds, test_ir_ds, lancedb_index_dir

In [ ]:
# Load the dataset
# Subset to 10 queries for a fast demonstration
train_data, test_data, lancedb_index_dir = load_ir_data(subset_queries=10)
print(f"Data loaded! Train pairs: {len(train_data)}, Test pairs: {len(test_data)}")

## Initializing the LlamaindexModel

DeepFix provides a `LlamaindexModel` wrapper that satisfies scikit-learn's estimator interface (implementing `fit`, `predict`, and `predict_proba`) over LlamaIndex retrieval workflows.

We initialize `LlamaindexModel` with our LanceDB vector index and fit the model:

In [ ]:
# Initialize and fit the LlamaindexModel
model = LlamaindexModel(
    dataset=train_data,
    load_if_exists=True,
    lancedb_index_dir=lancedb_index_dir,
    top_k=5,
    retrieval_mode="dense",
)
model.fit()
model_name = "llamaindex"

## Running DeepFix Diagnostic Analysis

Now we will run the automated diagnosis. The `DeepFixClient` will:
1. Ingest the datasets and the `LlamaindexModel`.
2. Trigger the Deepchecks diagnostic suites on the datasets to analyze metadata, data distribution, and text embeddings.
3. Call the model evaluator to measure precision, recall, and potential leakage or overfitting.
4. Synthesize all findings and provide a prioritized list of recommendations.

In [ ]:
# Run automated diagnosis with LlamaindexModel
result = client.get_diagnosis(
    train_data=train_data,
    test_data=test_data,
    model=model,
    model_name=model_name,
    language="english",
)

## Reviewing Diagnostic Results

Once analysis is complete, we can visualize the prioritized findings and recommendations in plain text using the `.to_text()` method.

In [ ]:
# Display summary of findings and recommended actions
result.to_text()